In [1]:
import numpy as np
import csv
import time
import os
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import skimage
import glob

In [2]:
section_id = 'S16-44514-E7U2'
selection_donor = "Myeloid cell"
selection_reciever = "Fibroblast"
BASE_DIR = '/home/aplante/aplante_diskmnt/mCRC/xenium_5k/Evan/morphcia/'

grid_size = 20 # microns
max_interaction_dist = 100 # microns

gene_start_index = 13

num_cells_expression_filter_cutoff = 100 # min cells expressing gene
mean_expression_filter_cutoff = 0.001 # mean expression of the cells that express the gene

abs_correlation_cutoff = 0.2 # correlation cutoff to keep pairs
mode = 'gene-gene_mask'

In [3]:
seurat_table = pd.read_csv(BASE_DIR + 'seurat_to_python/' + section_id + '.csv', low_memory=False)

In [6]:
output_dir = "morphcia/preprocess/"+section_id
if not os.path.isdir(output_dir):
    os.mkdir(output_dir)

In [4]:
kernal_size = int(max_interaction_dist/grid_size)

if (kernal_size % 1) != 0:
    raise Exception('max_interaction_dist must be an integer multiple of grid_size')
if (kernal_size % 2) == 0:
    raise Exception('max_interaction_dist must be an odd integer multiple of grid_size')


In [5]:
xcenter, ycenter = 6459, 3206
length = 800
x_start = xcenter - length
x_end = xcenter + length
y_start = ycenter - length
y_end = ycenter + length

spatial_subset = seurat_table[
            (seurat_table["x"] >= x_start) & (seurat_table["x"] < x_end) &
            (seurat_table["y"] >= y_start) & (seurat_table["y"] < y_end)
        ]

In [10]:
type_combos = list(itertools.combinations(np.unique(spatial_subset["broad_cell_type"]), 2))
tot_combos = len(type_combos)
q=0
for selection_donor, selection_reciever in type_combos:
    print()
    print(str(q), '/', str(tot_combos))
    q+=1
    # donor_cells = seurat_table[(seurat_table["broad_cell_type"] == selection_donor)]
    # reciever_cells = seurat_table[(seurat_table["broad_cell_type"] == selection_reciever)]
    
    donor_cells = spatial_subset[(spatial_subset["broad_cell_type"] == selection_donor)]
    reciever_cells = spatial_subset[(spatial_subset["broad_cell_type"] == selection_reciever)]
    
    x_min = np.floor(min(donor_cells["x"].min(), reciever_cells["x"].min()))
    x_max = np.ceil(max(donor_cells["x"].max(), reciever_cells["x"].max()))
    y_min = np.floor(min(donor_cells["y"].min(), reciever_cells["y"].min()))
    y_max = np.ceil(max(donor_cells["y"].max(), reciever_cells["y"].max()))
    
    x_values = np.arange(x_min - grid_size, x_max + grid_size, grid_size)
    y_values = np.arange(y_min - grid_size, y_max + grid_size, grid_size)
    
    gene_names = donor_cells.columns[gene_start_index:].to_numpy()
    
    gene_grid = np.zeros((2, gene_names.shape[0], x_values.shape[0], y_values.shape[0]))
    
    for i in range(len(x_values) - 1):
        percent_done = (i + 1) / x_values.shape[0] * 100
        print(f"Progress: {percent_done:5.1f}% ({i+1}/{len(x_values)})", end='\r')
        for j in range(len(y_values) - 1):
            x_start, x_end = x_values[i], x_values[i + 1]
            y_start, y_end = y_values[j], y_values[j + 1]
            
            donor_cells_in_grid = donor_cells[
                (donor_cells["x"] >= x_start) & (donor_cells["x"] < x_end) &
                (donor_cells["y"] >= y_start) & (donor_cells["y"] < y_end)
            ]
    
            reciever_cells_in_grid = reciever_cells[
                (reciever_cells["x"] >= x_start) & (reciever_cells["x"] < x_end) &
                (reciever_cells["y"] >= y_start) & (reciever_cells["y"] < y_end)
            ]
            
            if len(donor_cells_in_grid)>0:
                gene_grid[0,:,i,j] = donor_cells_in_grid.iloc[:, gene_start_index:].to_numpy().mean(axis=0)
    
            if len(reciever_cells_in_grid)>0:
                gene_grid[1,:,i,j] = reciever_cells_in_grid.iloc[:, gene_start_index:].to_numpy().mean(axis=0)

    np.save(output_dir+"/gene_grid_" + str(xcenter) + "_" + str(ycenter) + "_" + selection_donor + "_" + selection_reciever + ".npy", gene_grid)
    np.save(output_dir+"/gene_names_" + str(xcenter) + "_" + str(ycenter) + "_" + selection_donor + "_" + selection_reciever + ".npy", gene_names)


0 / 66
Progress:  98.8% (80/81)
1 / 66
Progress:  98.8% (80/81)
2 / 66
Progress:  98.8% (81/82)
3 / 66
Progress:  98.8% (79/80)
4 / 66
Progress:  98.8% (81/82)
5 / 66
Progress:  98.8% (79/80)
6 / 66
Progress:  98.8% (79/80)
7 / 66
Progress:  98.8% (81/82)
8 / 66
Progress:  98.8% (80/81)
9 / 66
Progress:  98.8% (80/81)
10 / 66
Progress:  98.8% (81/82)
11 / 66
Progress:  98.8% (80/81)
12 / 66
Progress:  98.8% (81/82)
13 / 66
Progress:  98.8% (80/81)
14 / 66
Progress:  98.8% (81/82)
15 / 66
Progress:  98.8% (80/81)
16 / 66
Progress:  98.8% (80/81)
17 / 66
Progress:  98.8% (81/82)
18 / 66
Progress:  98.8% (80/81)
19 / 66
Progress:  98.8% (81/82)
20 / 66
Progress:  98.8% (81/82)
21 / 66
Progress:  98.8% (81/82)
22 / 66
Progress:  98.8% (80/81)
23 / 66
Progress:  98.8% (81/82)
24 / 66
Progress:  98.8% (80/81)
25 / 66
Progress:  98.8% (80/81)
26 / 66
Progress:  98.8% (81/82)
27 / 66
Progress:  98.8% (80/81)
28 / 66
Progress:  98.8% (81/82)
29 / 66
Progress:  98.8% (81/82)
30 / 66
Progress:  